Group Members: Nicolas Banatt, Annanya Jain, James McDermott, Yanran Jia

In [165]:
!python --version

Python 3.13.7


In [166]:
%pip install pandas

Note: you may need to restart the kernel to use updated packages.


In [167]:
import pandas as pd

In [168]:
df_ag = pd.read_csv('AGData.csv') # ActiGraph accelerometer data
df_rrv = pd.read_csv('RRVData.csv') # Relative reinforcing value data

df_ag.head()

,Participant,ID,AG_Date,Completed,Gender,Group,Weight,Height,Race,Ethnicity,...,Avg_Daily_Light_Sa_Su_Min,Avg_Daily_Mod_Week_Min,Avg_Daily_Mod_M_F_Min,Avg_Daily_Mod_Sa_Su_Min,Avg_Daily_Vig_Week_Min,Avg_Daily_Vig_M_F_Min,Avg_Daily_Vig_Sa_Su_Min,Avg_Daily_Very_Vig_Week_Min,Avg_Daily_Very_Vig_M_F_Min,Avg_Daily_Very_Vig_Sa_Su_Min
0,407-0001,1,11-Dec-17,Y,M,control,NaN,NaN,White,Not Hispanic or Latino,...,264.5,22.0,26.0,13,3.0,4.0,0,0.0,0.0,0
1,407-0001,1,14-Dec-17,Y,M,control,129.4,196.0,White,Not Hispanic or Latino,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,407-0001,1,5-Mar-18,Y,M,control,NaN,NaN,White,Not Hispanic or Latino,...,275.5,47.0,47.0,47,4.0,5.0,0,0.0,0.0,0
3,407-0001,1,9-Apr-18,Y,M,control,NaN,NaN,White,Not Hispanic or Latino,...,252.5,40.0,38.0,45.5,0.0,0.0,0,0.0,0.0,0
4,407-0002,2,4-Dec-17,Y,M,control,NaN,NaN,White,Not Hispanic or Latino,...,79.5,8.0,8.0,6.5,0.0,0.0,0,0.0,0.0,0


In [169]:
print("AG shape:", df_ag.shape)
print("RRV shape:", df_rrv.shape)

AG shape: (295, 31)
RRV shape: (295, 55)


In [170]:
print(df_ag.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 295 entries, 0 to 294
Data columns (total 31 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   Participant                   283 non-null    object 
 1   ID                            295 non-null    int64  
 2   AG_Date                       278 non-null    object 
 3   Completed                     57 non-null     object 
 4   Gender                        280 non-null    object 
 5   Group                         260 non-null    object 
 6   Weight                        85 non-null     float64
 7   Height                        85 non-null     float64
 8   Race                          278 non-null    object 
 9   Ethnicity                     257 non-null    object 
 10  Age                           273 non-null    float64
 11  BMI                           85 non-null     float64
 12  Assmnt                        207 non-null    object 
 13  Avg_D

In [171]:
keep_cols = [
    "Participant","ID","Gender","Group","Race","Ethnicity",#"Age",
    "Assmnt",
    "Avg_Daily_Week_Min",
    "Avg_Daily_Sed_Week_Min",
    "Avg_Daily_Light_Week_Min",
    "Avg_Daily_Mod_Week_Min",
    "Avg_Daily_Vig_Week_Min",
    "Avg_Daily_Very_Vig_Week_Min"
]

df_ag = df_ag[keep_cols].copy()
print("Shape after column selection:", df_ag.shape)


Shape after column selection: (295, 13)


In [172]:
activity_cols = [
    "Avg_Daily_Week_Min",
    "Avg_Daily_Sed_Week_Min",
    "Avg_Daily_Light_Week_Min",
    "Avg_Daily_Mod_Week_Min",
    "Avg_Daily_Vig_Week_Min",
    "Avg_Daily_Very_Vig_Week_Min"
]

before = df_ag.shape[0]
df_ag = df_ag.dropna(subset=["Assmnt"] + activity_cols, how="all")
after = df_ag.shape[0]
print(f"Dropped {before - after} junk rows. New shape: {df_ag.shape}")

Dropped 88 junk rows. New shape: (207, 13)


In [173]:
df_ag["Assmnt"] = df_ag["Assmnt"].str.lower().str.strip()
mapping = {
    "baseline": "baseline",
    "1": "baseline",
    "endpsttr": "endposttr",
    "endposttr": "endposttr",
    "6": "endposttr",
    "postwash": "postwash",
    "10": "postwash",
}

df_ag["Assmnt"] = df_ag["Assmnt"].replace(mapping)
print("Assmnt value counts after cleaning:")
print(df_ag["Assmnt"].value_counts(dropna=False))

Assmnt value counts after cleaning:
Assmnt
baseline     84
pstwash      64
endposttr    59
Name: count, dtype: int64


In [174]:
import numpy as np

df_ag["Group"] = df_ag["Group"].str.lower().str.strip()

group_mapping = {
    "control": 0,
    "experimental": 1,
    "": np.nan,
    None: np.nan
}
df_ag["Group"] = df_ag["Group"].replace(group_mapping)
print("Group unique values:", df_ag["Group"].unique())

Group unique values: [ 0.  1. nan]


C:\Users\minec\AppData\Local\Temp\ipykernel_10180\784456824.py:11: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_ag["Group"] = df_ag["Group"].replace(group_mapping)


In [175]:
for col in activity_cols:
    df_ag[col] = pd.to_numeric(df_ag[col], errors="coerce")

print(df_ag[activity_cols].dtypes)


Avg_Daily_Week_Min             float64
Avg_Daily_Sed_Week_Min         float64
Avg_Daily_Light_Week_Min       float64
Avg_Daily_Mod_Week_Min         float64
Avg_Daily_Vig_Week_Min         float64
Avg_Daily_Very_Vig_Week_Min    float64
dtype: object


In [176]:
before = df_ag.shape[0]
df_ag = df_ag.dropna(subset=activity_cols, how="any")
after = df_ag.shape[0]
print(f"Dropped {before - after} rows missing activity data. New shape: {df_ag.shape}")

Dropped 0 rows missing activity data. New shape: (207, 13)


In [177]:
df_ag["Gender"] = df_ag["Gender"].fillna(df_ag["Gender"].mode()[0])
df_ag["Race"] = df_ag["Race"].fillna(df_ag["Race"].mode()[0])
df_ag["Ethnicity"] = df_ag["Ethnicity"].fillna(df_ag["Ethnicity"].mode()[0])

In [178]:
#df_ag["Age"] = df_ag["Age"].fillna(df_ag["Age"].median())
df_ag["Group"] = df_ag["Group"].fillna(df_ag["Group"].mode()[0])

In [179]:
print(df_ag.isna().sum())

Participant                    0
ID                             0
Gender                         0
Group                          0
Race                           0
Ethnicity                      0
Assmnt                         0
Avg_Daily_Week_Min             0
Avg_Daily_Sed_Week_Min         0
Avg_Daily_Light_Week_Min       0
Avg_Daily_Mod_Week_Min         0
Avg_Daily_Vig_Week_Min         0
Avg_Daily_Very_Vig_Week_Min    0
dtype: int64


In [180]:
print("Final shape:", df_ag.shape)
df_ag.head()

Final shape: (207, 13)


,Participant,ID,Gender,Group,Race,Ethnicity,Assmnt,Avg_Daily_Week_Min,Avg_Daily_Sed_Week_Min,Avg_Daily_Light_Week_Min,Avg_Daily_Mod_Week_Min,Avg_Daily_Vig_Week_Min,Avg_Daily_Very_Vig_Week_Min
0,407-0001,1,M,0.0,White,Not Hispanic or Latino,baseline,910.0,629.0,256.0,22.0,3.0,0.0
2,407-0001,1,M,0.0,White,Not Hispanic or Latino,endposttr,824.0,521.0,252.0,47.0,4.0,0.0
3,407-0001,1,M,0.0,White,Not Hispanic or Latino,pstwash,848.0,579.0,229.0,40.0,0.0,0.0
4,407-0002,2,M,0.0,White,Not Hispanic or Latino,baseline,680.0,546.0,127.0,8.0,0.0,0.0
6,407-0002,2,M,0.0,White,Not Hispanic or Latino,endposttr,622.0,510.0,102.0,11.0,0.0,0.0


In [181]:
df_ag.to_csv("AGData_clean.csv", index=False)

In [182]:
# Now, clean RRV data
df_rrv.head()

,Participant,ID,RRV_Date,Completed,Gender,Group,Weight,Height,Race,Ethnicity,...,breq3Intrich,breq3Exter,breq3Exterch,breq3Integ,breq3Integch,breq3Ident,breq3Identch,breq3Intro,breq3Introch,breq3Amoti
0,407-0001,1,14-Dec-17,Y,M,NaN,129.4,196.0,White,Not Hispanic or Latino,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,407-0001,1,18-Dec-17,Y,M,control,124.7,196.0,White,Not Hispanic or Latino,...,NaN,0.0,NaN,14.0,NaN,12.0,NaN,4.0,NaN,0.0
2,407-0001,1,20-Feb-18,Y,M,control,121.6,NaN,White,Not Hispanic or Latino,...,4.0,0.0,0.0,13.0,-1.0,12.0,0.0,4.0,0.0,0.0
3,407-0001,1,28-Mar-18,Y,M,control,123.6,NaN,White,Not Hispanic or Latino,...,0.0,0.0,0.0,10.0,-4.0,12.0,0.0,4.0,0.0,0.0
4,407-0002,2,8-Dec-17,Y,M,NaN,109.1,184.2,White,Not Hispanic or Latino,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [183]:
df_rrv.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 295 entries, 0 to 294
Data columns (total 55 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Participant    283 non-null    object 
 1   ID             295 non-null    int64  
 2   RRV_Date       276 non-null    object 
 3   Completed      46 non-null     object 
 4   Gender         284 non-null    object 
 5   Group          198 non-null    object 
 6   Weight         278 non-null    float64
 7   Height         151 non-null    float64
 8   Race           283 non-null    object 
 9   Ethnicity      261 non-null    object 
 10  Age            276 non-null    float64
 11  BMI            278 non-null    float64
 12  Assmnt         198 non-null    object 
 13  time           198 non-null    float64
 14  RRV_Pgoal      195 non-null    object 
 15  RRV_PPmax      195 non-null    float64
 16  RRVPpmaxch     132 non-null    float64
 17  RRV_Presp      195 non-null    float64
 18  RRV_Prespc

In [184]:
keep_cols = [
    "Participant", "ID", "Assmnt",
    "Height", "Age", "BMI",
    "RRV_Pgoal", "RRV_Agoal", "RRVscore","RRVch",
    "oesPos", "oesNeg",
    "pretieqPref", "pretieqTole"
]

df_rrv = df_rrv[keep_cols].copy()
print("Shape after column selection:", df_rrv.shape)
print(df_rrv.isna().sum())

Shape after column selection: (295, 14)
Participant     12
ID               0
Assmnt          97
Height         144
Age             19
BMI             17
RRV_Pgoal      100
RRV_Agoal      100
RRVscore       102
RRVch          163
oesPos         163
oesNeg         163
pretieqPref    100
pretieqTole    100
dtype: int64


In [185]:
df_rrv = df_rrv.dropna(subset=["Participant","ID","Assmnt"])

df_rrv["Assmnt"] = df_rrv["Assmnt"].str.lower().str.strip()
mapping = {
    "baseline": "baseline",
    "1": "baseline",
    "endpsttr": "endposttr",
    "endposttr": "endposttr",
    "6": "endposttr",
    "postwash": "pstwash",
    "10": "postwash",
}

df_rrv["Assmnt"] = df_rrv["Assmnt"].replace(mapping)
print("Assmnt value counts after cleaning:")
print(df_rrv["Assmnt"].value_counts(dropna=False))

Assmnt value counts after cleaning:
Assmnt
baseline     66
endposttr    66
pstwash      66
Name: count, dtype: int64


In [186]:
# Fill height with any one of the participant's heights, else replace with median height
df_rrv["Height"] = df_rrv.groupby("Participant")["Height"].transform(
    lambda x: x.fillna(x.dropna().iloc[0] if not x.dropna().empty else None)
)
df_rrv["Height"] = df_rrv["Height"].fillna(df_rrv["Height"].median())


df_rrv["Age"] = df_rrv.groupby("Participant")["Age"].transform(lambda x: x.ffill().bfill())
df_rrv["Age"] = df_rrv["Age"].fillna(df_rrv["Age"].median())

df_rrv["BMI"] = df_rrv.groupby("Participant")["BMI"].transform(lambda x: x.fillna(x.mean()))
df_rrv["BMI"] = df_rrv["BMI"].fillna(df_rrv["BMI"].median())

In [187]:
for col in ["RRV_Pgoal","RRV_Agoal"]:
    if df_rrv[col].isnull().any():
        mode_val = df_rrv[col].mode()[0] if not df_rrv[col].mode().empty else "Unknown"
        df_rrv[col] = df_rrv[col].fillna(mode_val)

for col in ["RRVscore","oesPos","oesNeg","pretieqPref","pretieqTole"]:
    df_rrv[col] = df_rrv.groupby("Participant")[col].transform(lambda x: x.fillna(x.mean()))
    df_rrv[col] = df_rrv[col].fillna(df_rrv[col].mean())

mask_baseline = (df_rrv["Assmnt"].str.lower() == "baseline") & (df_rrv["RRVch"].isna())
df_rrv.loc[mask_baseline, "RRVch"] = 0
df_rrv = df_rrv.dropna(subset=["RRVch"])

df_rrv.head()

,Participant,ID,Assmnt,Height,Age,BMI,RRV_Pgoal,RRV_Agoal,RRVscore,RRVch,oesPos,oesNeg,pretieqPref,pretieqTole
1,407-0001,1,baseline,196.0,32.0,32.5,ellip,read,0.50,0.00,90.0,3.33,30.0,24.0
2,407-0001,1,endposttr,196.0,33.0,31.7,tmill,read,1.00,0.50,91.0,13.33,32.0,23.0
3,407-0001,1,pstwash,196.0,33.0,32.2,tmill,read,0.75,0.25,90.5,8.33,30.0,39.0
5,407-0002,2,baseline,184.2,33.0,32.1,cycle,wrdgms,0.00,0.00,83.0,48.33,26.0,20.0
6,407-0002,2,endposttr,184.2,33.0,32.0,ellip,cwrdpz,0.00,0.00,71.0,46.17,23.0,19.0


In [188]:
print(df_rrv.isna().sum())

df_rrv.to_csv("RRVData_clean.csv", index=False)

Participant    0
ID             0
Assmnt         0
Height         0
Age            0
BMI            0
RRV_Pgoal      0
RRV_Agoal      0
RRVscore       0
RRVch          0
oesPos         0
oesNeg         0
pretieqPref    0
pretieqTole    0
dtype: int64


In [189]:
# For later, change RRVch (our target) to binary label for classification
# df["RRV_label"] = (df["RRVch"] > 0).astype(int)

# Join the datasets
ag = pd.read_csv("AGData_clean.csv")
rrv = pd.read_csv("RRVData_clean.csv")

merged = pd.merge(
    ag,
    rrv,
    on=["Participant", "ID", "Assmnt"],
    how="inner"   # only keep rows that exist in both datasets
)

merged.to_csv("MergedData_clean.csv", index=False)
merged.head()

,Participant,ID,Gender,Group,Race,Ethnicity,Assmnt,Avg_Daily_Week_Min,Avg_Daily_Sed_Week_Min,Avg_Daily_Light_Week_Min,...,Age,BMI,RRV_Pgoal,RRV_Agoal,RRVscore,RRVch,oesPos,oesNeg,pretieqPref,pretieqTole
0,407-0001,1,M,0.0,White,Not Hispanic or Latino,baseline,910.0,629.0,256.0,...,32.0,32.5,ellip,read,0.50,0.00,90.0,3.33,30.0,24.0
1,407-0001,1,M,0.0,White,Not Hispanic or Latino,endposttr,824.0,521.0,252.0,...,33.0,31.7,tmill,read,1.00,0.50,91.0,13.33,32.0,23.0
2,407-0001,1,M,0.0,White,Not Hispanic or Latino,pstwash,848.0,579.0,229.0,...,33.0,32.2,tmill,read,0.75,0.25,90.5,8.33,30.0,39.0
3,407-0002,2,M,0.0,White,Not Hispanic or Latino,baseline,680.0,546.0,127.0,...,33.0,32.1,cycle,wrdgms,0.00,0.00,83.0,48.33,26.0,20.0
4,407-0002,2,M,0.0,White,Not Hispanic or Latino,endposttr,622.0,510.0,102.0,...,33.0,32.0,ellip,cwrdpz,0.00,0.00,71.0,46.17,23.0,19.0
